# ScoutTrainer — GPU pipeline on Colab

Runs the heavy perception stage on Colab's free GPU (minutes instead of hours), then hands you a zip of results to browse in your **local** dashboard.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

### About YouTube URLs on Colab
YouTube blocks downloads from datacenter IPs ("Sign in to confirm you're not a bot"), so **URLs usually fail here even though they work on your home machine**. Recommended path: download the match video on your PC, then upload the `.mp4` in step 3A below. Steps 3B/3C are fallbacks if you'd rather try the URL.

In [ ]:
# 1) Upload the project zip (zip the scout-agent folder WITHOUT .venv, data/, .git)
#    Running this cell shows a 'Choose Files' button — pick scout-agent.zip from your PC.
from google.colab import files
up = files.upload()
!rm -rf /content/scout && mkdir -p /content/scout
import zipfile, pathlib
zipfile.ZipFile(next(iter(up))).extractall('/content/scout')
root = next(p.parent for p in pathlib.Path('/content/scout').rglob('pyproject.toml'))
%cd {root}
!nvidia-smi -L

In [ ]:
# 2) Install deps (Colab already has torch+CUDA and ffmpeg). Deno = JS runtime for yt-dlp.
!pip -q install -e ".[perception]" 2>&1 | tail -1
!curl -fsSL https://deno.land/install.sh | DENO_INSTALL=/usr/local sh -s -- -y >/dev/null 2>&1
!deno --version | head -1
import torch; print('CUDA available:', torch.cuda.is_available())

## 3) Choose ONE source cell (A recommended)

In [ ]:
# 3A) RECOMMENDED — upload a video file from your PC (no YouTube blocking)
from google.colab import files
import pathlib, shutil, re
vid = files.upload()                      # 'Choose Files' button appears below
name = next(iter(vid))
safe = re.sub(r'[^A-Za-z0-9._-]+', '_', name)   # spaces break shell arguments
dest = pathlib.Path('data/videos'); dest.mkdir(parents=True, exist_ok=True)
shutil.move(name, dest / safe)
SOURCE = ['--file', str(dest / safe)]
print('using', dest / safe)

In [ ]:
# 3B) Try a YouTube URL directly (often blocked on Colab — see note at top)
SOURCE = ['--url', 'https://www.youtube.com/watch?v=E9GLiV_jfro']  # <-- change me

In [ ]:
# 3C) YouTube URL + cookies (unblocks most bot checks)
#     Export cookies.txt from a logged-in browser using a 'Get cookies.txt' extension,
#     then upload it here. Use a throwaway Google account — cookies are credentials.
import os
from google.colab import files
ck = files.upload()                       # pick cookies.txt
os.environ['SCOUT_YTDLP_COOKIES'] = '/content/' + next(iter(ck))
SOURCE = ['--url', 'https://www.youtube.com/watch?v=E9GLiV_jfro']  # <-- change me

## 4a) Pitch reference points (needed by the `project` stage)

The pipeline needs to know how video pixels map to pitch metres. Run the next cell to save a frame, read 4 pixel coordinates off the plot axes (pitch corners are best; penalty-box corners work too), then fill them into the cell after it.

Pitch coordinate system: origin `[0, 0]` = left-bottom corner as seen from the camera, pitch is 100 m long x 64 m wide, so the four corners are `[0,0]`, `[100,0]`, `[100,64]`, `[0,64]`.

In [ ]:
# 4a-i) Show a frame with pixel axes — note the x,y of 4 known pitch landmarks
import cv2, glob, matplotlib.pyplot as plt
vid = sorted(glob.glob('data/matches/*/video.mp4'))[-1]
cap = cv2.VideoCapture(vid)
cap.set(cv2.CAP_PROP_POS_FRAMES, int(cap.get(cv2.CAP_PROP_FRAME_COUNT) * 0.5))  # mid-video frame
ok, frame = cap.read(); cap.release()
plt.figure(figsize=(16, 9))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.grid(color='yellow', alpha=0.4); plt.xticks(range(0, frame.shape[1], 50), rotation=90, fontsize=6)
plt.yticks(range(0, frame.shape[0], 50), fontsize=6); plt.show()
print('frame size (w,h):', frame.shape[1], frame.shape[0])

In [ ]:
# 4a-ii) Fill in the pixel coords you read above, then run to write refs.json
import json
refs = {'points': [
    {'px': [112, 596], 'pitch': [0, 0]},      # <-- left-bottom corner
    {'px': [1163, 601], 'pitch': [100, 0]},   # <-- right-bottom corner
    {'px': [986, 82],  'pitch': [100, 64]},   # <-- right-top corner
    {'px': [297, 80],  'pitch': [0, 64]},     # <-- left-top corner
]}
open('refs.json', 'w').write(json.dumps(refs))
REF_POINTS = 'refs.json'
print('wrote refs.json')

In [ ]:
# 4) Optional extras — leave as '' to skip (upload via the folder icon in the left sidebar)
REF_POINTS = ''   # refs.json: 4 pixel->pitch points; without it the 'project' stage stops
ROSTER = ''       # roster.csv: jersey_number,name

In [ ]:
# 5) Run the pipeline on GPU (yolov8x is auto-selected when CUDA is available)
import os, shlex
os.environ['SCOUT_DEVICE'] = 'cuda'
args = list(SOURCE)
if REF_POINTS: args += ['--ref-points', REF_POINTS]
if ROSTER: args += ['--roster', ROSTER]
cmd = shlex.join(args)   # quotes any path containing spaces
!python -m scout.pipeline {cmd}

In [ ]:
# 6) Download results — extract into your LOCAL project's data/ folder,
#    then run `streamlit run app/dashboard.py` locally to browse ratings
!cd data && zip -qr /content/scout_results.zip . -x 'videos/*' 'uploads/*'
from google.colab import files
files.download('/content/scout_results.zip')